# Loss Function Experiments: LSTM Volatility Forecasting

Benchmarks three training objectives for the LSTM model on BTC daily realized volatility:

| # | Loss | Objective |
|---|---|---|
| 1 | **MSE** | Minimize squared magnitude error |
| 2 | **Directional MSE** | MSE + penalty for wrong directional prediction |
| 3 | **QLIKE** | Proportional error — standard loss in financial volatility research |

All experiments share the same architecture (2-layer LSTM, hidden=64, dropout=0.2),
data pipeline, and train/val/test split (2018–2022 / 2023 / 2024).

## Background: LSTM for Volatility Forecasting

LSTM (Long Short-Term Memory) extends RNNs with a **cell state** — a gradient highway
that lets information persist across long sequences without vanishing. Three gates control
information flow:

- **Forget gate**: discards irrelevant parts of the prior cell state
- **Input gate**: writes new information into the cell state
- **Output gate**: exposes a portion of the cell state as the hidden state

### Why these gates matter for volatility

The forget gate lets the model drop stale market regimes and focus on meaningful volatility
events — sudden spikes or crashes that persist for weeks before mean-reverting. The input
gate controls whether the current day's price action is informative enough to update the
model's internal regime representation.

### Feature choice

- **`log_returns`**: stationary, scale-invariant representation of daily price moves.
  Raw prices are non-stationary and not comparable across the 2018–2024 range.
- **`volatility`** (30-day realized): captures the current turbulence regime and provides
  context that a single return observation cannot supply.

### Lookback window (30 days)

A longer window lets the model judge whether a move is historically unusual — similar to a
long-memory statistical baseline. A shorter window is more reactive but greedy, seeing only
the immediate regime. 30 days balances regime context with responsiveness for BTC, which
exhibits strong volatility clustering (GARCH α+β ≈ 0.92).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import TensorDataset, DataLoader
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

## Data Pipeline

In [ ]:
btc_df = pd.read_csv('../data/btc_data.csv', index_col='Date', parse_dates=True)

train = btc_df.loc['2018-01-31':'2022-12-31']
val   = btc_df.loc['2023-01-01':'2023-12-31']
test  = btc_df.loc['2024-01-01':'2024-12-31']

print(f'Train: {len(train)} days | Val: {len(val)} days | Test: {len(test)} days')

In [ ]:
def scale_data(train_df, val_df, test_df):
    # Fit all scalers on train only; apply to val and test
    log_scaler    = StandardScaler()
    vol_scaler    = StandardScaler()
    target_scaler = StandardScaler()

    def build_df(df, log_data, vol_data, target_data):
        return pd.concat([
            pd.DataFrame(log_data,           index=df.index, columns=['log_returns']),
            pd.DataFrame(vol_data,           index=df.index, columns=['volatility']),
            pd.Series(target_data.flatten(), index=df.index, name='realized_variance'),
        ], axis=1)

    scaled_train = build_df(
        train_df,
        log_scaler.fit_transform(train_df[['log_returns']]),
        vol_scaler.fit_transform(train_df[['volatility']]),
        target_scaler.fit_transform(train_df[['realized_variance']]),
    )
    scaled_val = build_df(
        val_df,
        log_scaler.transform(val_df[['log_returns']]),
        vol_scaler.transform(val_df[['volatility']]),
        target_scaler.transform(val_df[['realized_variance']]),
    )
    scaled_test = build_df(
        test_df,
        log_scaler.transform(test_df[['log_returns']]),
        vol_scaler.transform(test_df[['volatility']]),
        target_scaler.transform(test_df[['realized_variance']]),
    )
    return scaled_train, scaled_val, scaled_test, target_scaler


def create_sequences(df, window_size=30):
    X_data = df[['log_returns', 'volatility']].values
    y_data = df['realized_variance'].values
    X = [X_data[i:i + window_size] for i in range(len(df) - window_size)]
    y = [y_data[i + window_size]   for i in range(len(df) - window_size)]
    return (
        torch.tensor(np.array(X), dtype=torch.float32),
        torch.tensor(np.array(y), dtype=torch.float32),
    )

In [ ]:
WINDOW = 30
BATCH  = 128

scaled_train, scaled_val, scaled_test, target_scaler = scale_data(train, val, test)

X_train, y_train = create_sequences(scaled_train, WINDOW)
X_val,   y_val   = create_sequences(scaled_val,   WINDOW)
X_test,  y_test  = create_sequences(scaled_test,  WINDOW)

print(f'X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')

train_loader = DataLoader(TensorDataset(X_train, y_train), shuffle=True, batch_size=BATCH)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),                batch_size=BATCH)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),               batch_size=BATCH)

## Model & Training Infrastructure

In [ ]:
from models.lstm import LSTMModel

def make_model():
    return LSTMModel(input_size=2, hidden_size=64, num_layers=2, dropout=0.2)

def make_optimizer(model, lr=1e-3):
    return torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
def train_and_validate(model, train_loader, val_loader, optimizer, loss_fn,
                       save_path, num_epochs=50):
    # loss_fn signature: (pred, X_batch, y_batch) -> scalar tensor
    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path))
        model.eval()
        print(f'Loaded from {save_path}')
        return model

    print(f'Training {os.path.basename(save_path)}...')
    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        model.train()
        running_train = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.float(), y_b.float().view(-1, 1)
            loss = loss_fn(model(X_b), X_b, y_b)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_train += loss.item()

        model.eval()
        running_val = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.float(), y_b.float().view(-1, 1)
                running_val += loss_fn(model(X_b), X_b, y_b).item()

        avg_train = running_train / len(train_loader)
        avg_val   = running_val   / len(val_loader)
        train_losses.append(avg_train)
        val_losses.append(avg_val)

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:>3}/{num_epochs}  '
                  f'train={avg_train:.6f}  val={avg_val:.6f}')

    torch.save(model.state_dict(), save_path)

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(train_losses, label='Train')
    ax.plot(val_losses,   label='Val')
    ax.set(xlabel='Epoch', ylabel='Loss',
           title=f'Loss curve — {os.path.basename(save_path)}')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return model

In [ ]:
def qlike_np(actual, pred, eps=1e-10):
    actual = np.clip(actual, eps, None)
    pred   = np.clip(pred,   eps, None)
    return ((actual / pred) - np.log(actual / pred) - 1).mean()


def evaluate(model, test_loader, scaled_test_df, target_scaler, window=30):
    model.eval()
    preds = []
    with torch.no_grad():
        for X_b, _ in test_loader:
            preds.append(model(X_b.float()))

    pred_scaled   = torch.cat(preds).cpu().numpy()
    actual_scaled = scaled_test_df['realized_variance'].iloc[window:].values.reshape(-1, 1)

    pred_real   = target_scaler.inverse_transform(pred_scaled).flatten()
    actual_real = target_scaler.inverse_transform(actual_scaled).flatten()
    idx         = scaled_test_df.index[window:]

    results = pd.DataFrame({'actual': actual_real, 'forecast': pred_real}, index=idx)

    mse = mean_squared_error(results['actual'], results['forecast'])
    mae = mean_absolute_error(results['actual'], results['forecast'])
    ql  = qlike_np(results['actual'].values, results['forecast'].values)

    print(f'MSE: {mse:.2e}  MAE: {mae:.5f}  QLIKE: {ql:.5f}')

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(results.index, results['actual'],   label='Actual')
    ax.plot(results.index, results['forecast'], label='Forecast', alpha=0.8)
    ax.set(ylabel='Realized Variance', title='2024 Test Set')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return results, {'mse': mse, 'mae': mae, 'qlike': ql}

## Loss Functions

All three loss functions share the signature `loss_fn(pred, X_batch, y_batch)` so the
same training loop handles every experiment without modification.

In [ ]:
_mse_fn = nn.MSELoss()


def mse_loss(pred, X_batch, y_batch):
    return _mse_fn(pred, y_batch)


def directional_mse_loss(pred, X_batch, y_batch, lam=0.1):
    yesterday = X_batch[:, -1, 1].view(-1, 1)  # last timestep volatility (feature index 1)
    # tanh(x * 10) approximates sign(x) with a valid gradient
    pred_dir   = torch.tanh((pred    - yesterday) * 10)
    actual_dir = torch.tanh((y_batch - yesterday) * 10)
    # Product is ~+1 when directions match, ~-1 when opposite
    # Negate so that correct direction lowers the loss
    dir_penalty = (-pred_dir * actual_dir).mean()
    return _mse_fn(pred, y_batch) + lam * dir_penalty


def qlike_loss(pred, X_batch, y_batch, eps=1e-10):
    # Note: StandardScaler can produce negative target values.
    # clamp(min=eps) ensures numerical stability but QLIKE is only
    # interpretable as a proper scoring rule on positive (unscaled) targets.
    actual = torch.clamp(y_batch, min=eps)
    pred   = torch.clamp(pred,    min=eps)
    return ((actual / pred) - torch.log(actual / pred) - 1).mean()

## Experiment 1: MSE Loss

Baseline. MSE penalizes squared magnitude error, so the Bayes-optimal prediction under
MSE is the conditional mean of the target. For a smooth, highly autocorrelated signal like
30-day realized volatility, this frequently collapses to predicting a near-constant value
close to the training mean — a failure mode known as **mean collapse**.

In [ ]:
model_mse = make_model()
opt_mse   = make_optimizer(model_mse)

model_mse = train_and_validate(
    model_mse, train_loader, val_loader, opt_mse, mse_loss,
    save_path='../models/mse_lstm_model.pth',
)
results_mse, metrics_mse = evaluate(model_mse, test_loader, scaled_test, target_scaler)

### Analysis: Mean Collapse

The model outputs a nearly constant value (~0.0056 scaled) regardless of the input window.
A direct test: if predictions correlate strongly with the *prior day's actual*, the model
is not forecasting — it is copying yesterday's value (a naive lag-1 forecast).

In [ ]:
lag1_corr = results_mse['forecast'].corr(results_mse['actual'].shift(1))
print(f'Correlation(forecast, lag-1 actual): {lag1_corr:.4f}')
# Near 1.0 confirms the model learned a near-constant lag-1 forecast

## Experiment 2: Directional MSE Loss

Adding a directional penalty encourages the model to predict the correct *sign* of the
day-over-day change, not just the correct magnitude:

$$\mathcal{L} = \text{MSE}(\hat{y},\, y) - \lambda \cdot \overline{\tanh\!\left((\hat{y} - y_{t-1})\times 10\right) \cdot \tanh\!\left((y - y_{t-1})\times 10\right)}$$

`tanh` with slope ×10 approximates `sign()` while remaining differentiable. The product is
+1 when both move in the same direction and −1 when they diverge; negating it means correct
direction reduces the loss. λ=0.1 balances the two terms.

**Reference point**: `X_batch[:, -1, 1]` — the last timestep's scaled volatility — is used
as the prior day's value for computing directional change.

In [ ]:
model_dir = make_model()
opt_dir   = make_optimizer(model_dir)

model_dir = train_and_validate(
    model_dir, train_loader, val_loader, opt_dir, directional_mse_loss,
    save_path='../models/directional_lstm_model.pth',
)
results_dir, metrics_dir = evaluate(model_dir, test_loader, scaled_test, target_scaler)

### Analysis: Directional MSE

The directional penalty breaks mean collapse — predictions now vary day-to-day. The tradeoffs:

- Directional accuracy improves modestly (~52% vs ~34% with pure MSE, essentially random in both cases)
- MSE and MAE worsen substantially — the model sacrifices magnitude accuracy to align directional signals

The limited improvement reflects a structural constraint: **30-day rolling volatility changes
by only 1/30th of the window each day** (one observation in, one out). Day-over-day changes
are tiny relative to the signal level, making directional accuracy a high-variance and
unreliable metric for this target.

## Experiment 3: QLIKE Loss

QLIKE (Quasi-Likelihood) is the standard evaluation loss in financial econometrics for
volatility models. It measures *proportional* error rather than absolute error:

$$\text{QLIKE}(y,\,\hat{y}) = \frac{y}{\hat{y}} - \log\frac{y}{\hat{y}} - 1$$

Properties:
- Global minimum at $\hat{y} = y$ (zero loss only at perfect prediction)
- Asymmetric: penalizes **underestimation more than overestimation**, appropriate for
  risk — predicting lower volatility than realized is more dangerous than the reverse
- Requires strictly positive predictions; `Softplus` output in `LSTMModel` guarantees this

> **Note on scaling**: `StandardScaler` can produce negative target values (variance is
> always positive, but the scaled version may not be). `torch.clamp(min=eps)` prevents
> numerical errors, but QLIKE is only a proper scoring rule on positive unscaled targets.
> This experiment is exploratory — in production, use a log-transform of the target instead.

In [ ]:
model_qlike = make_model()
opt_qlike   = make_optimizer(model_qlike)

model_qlike = train_and_validate(
    model_qlike, train_loader, val_loader, opt_qlike, qlike_loss,
    save_path='../models/qlike_lstm_model.pth',
)
results_qlike, metrics_qlike = evaluate(model_qlike, test_loader, scaled_test, target_scaler)

### Analysis: QLIKE

QLIKE training produces the same mean-collapse pattern as MSE — predictions converge to a
near-constant (~0.0378 scaled). The proportional penalty is minimized by predicting a
constant near the geometric mean of the target; the model finds this locally optimal
solution rather than learning input-conditional patterns.

Collapse is **not loss-function-specific** — it is caused by the smoothness of the 30-day
rolling target. Day-over-day variation is too small for the LSTM hidden state to add
predictive value beyond the unconditional mean.

## Summary

### Results

| Loss | Collapses? | Notes |
|---|---|---|
| MSE | Yes | Constant output ≈ training mean; lag-1 correlation ≈ 0.97 |
| Directional MSE | No | Breaks collapse; modest directional gain, worse MSE/MAE |
| QLIKE | Yes | Constant output ≈ geometric mean; same structural failure |

### Root cause

High autocorrelation of the 30-day rolling volatility target: consecutive days share 29 of
30 observations, so the lag-1 prediction is near-optimal under any smooth loss. The LSTM
provides little additional signal over the unconditional mean.

### Potential next steps

- **Different target**: raw squared log returns or 5-day realized variance — higher
  day-to-day variation gives the model more signal
- **Additional features**: volume, funding rate, or on-chain metrics to provide
  input-conditional signal the return/vol series alone does not supply
- **Log-transform target**: `log(realized_variance)` is still smooth but removes the
  scaling issue for QLIKE and gives a more Gaussian target distribution